<a href="https://colab.research.google.com/github/alexgufron-unnes/Smoothing-RandomForest-and-GVOA/blob/main/Experiment_JIKI_NLP_References_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dirichlet Smoothing and Adapted Vulture Search for Random Forest Sentiment Classification

**Alex Gufron¹*, Florentina Yuni Arini², Nizirwan Anwar³, Alamsyah¹**

1 Faculty of Mathematics and Natural Sciences, Universitas Negeri Semarang, Semarang 50229, Indonesia

2 Department of Computer Science, Faculty of Mathematics and Natural Sciences, Universitas Negeri Semarang, Sekaran, Gunungpati, Semarang 50229, Indonesia

3 Informatics Engineering Study Program, Universitas Esa Unggul, Jakarta, Indonesia

*Corresponding author:* alexgufron@students.unnes.ac.id

*Florentina Yuni Arini:* floyuna@mail.unnes.ac.id

Author-identified replication notebook for the JIKI manuscript. Select **Runtime > Run all** in a fresh Google Colab runtime. The public corpus snapshot needs no authentication; package installation requires internet. This notebook implements a fixed exploratory experiment, not a confirmatory evaluation. The test partition was already inspected during earlier development. Adapted vulture search is not a verified canonical GVOA implementation.

Probability smoothing addresses sparse counts in language models, but its benefit does not necessarily transfer to tree-based text classification. This study examined Dirichlet-smoothed features and adapted vulture search in an exploratory Indonesian sentiment experiment. A public telecommunications corpus contained 300 records; deterministic cleaning and deduplication retained 294 documents. A stratified split assigned 235 documents to training and 59 to testing. Three configurations shared unigram-bigram features: a random forest with relative-frequency features, the same forest with fixed Dirichlet smoothing, and a smoothed forest selected through three-fold training cross-validation (CV). The search evaluated 22 unique configurations. The baseline achieved accuracy 0.8136 and macro F1 0.8055. Fixed smoothing reduced macro F1 to 0.5743; adapted search increased it to 0.7065, recovering 0.1322 without surpassing the baseline. Training-text exploration identified a correlation of 0.9940 between token count and unique-token count. The contribution is a reproducible classification evaluation with explicit data provenance, matched representation, and prediction-level evaluation. The reused test split and unverified equivalence to canonical Griffon Vultures Optimization Algorithm restrict the findings to exploratory evidence. The results do not establish an advantage for the complete optimization pipeline.

## Author identities and manuscript status

Author order: Alex Gufron, Florentina Yuni Arini, Nizirwan Anwar, Alamsyah. Author affiliations and contact details follow the supplied source documents. Alex Gufron is the corresponding author. Alamsyah shares affiliation 1. The contribution statement remains a proposed allocation for author verification. This identified version follows the author request; JIKI requires an anonymous first-submission version.

The corpus contains original Indonesian text for sentiment classification. Source URLs, institution names and original corpus bytes remain unchanged. All explanations, code comments, messages, labels and generated result-file names use English. A cited Indonesian article title appears as an explicitly marked English translation; its source URL remains unchanged. The RIS contains one unpublished manuscript record and nine cited studies.


In [ ]:
%pip -q install numpy==2.3.5 pandas==2.2.3 scikit-learn==1.8.0 scipy==1.17.0 matplotlib

## 1 Dataset and source verification

Dataset: [owner repository](https://github.com/rizalespe/Dataset-Sentimen-Analisis-Bahasa-Indonesia). [Pinned CSV](https://raw.githubusercontent.com/rizalespe/Dataset-Sentimen-Analisis-Bahasa-Indonesia/1c8d2c99f281cf51e660378a06cad1fd3f1e371d/dataset_tweet_sentiment_cellular_service_provider.csv). [Source study](https://j-ptiik.ub.ac.id/index.php/j-ptiik/article/view/628).

Commit: `1c8d2c99f281cf51e660378a06cad1fd3f1e371d`.

SHA-256: `6b0851bb960b2bc09bf1e92d3b735fdd2f7e19ac0f1d54c43829018095b188d6`.

The following Base64 value is the original public file, not generated observations. The loader checks its digest and aborts on mismatch. An optional local file must match the same digest. Labels remain source labels. After cleaning, six duplicate rows are removed and 294 documents remain. Class counts and missingness are recomputed on each run. Review repository data-use conditions before redistribution or publication. Attribution to the source study does not imply independent label validation.

In [ ]:
# Verified original public corpus snapshot.
ORIGINAL_CORPUS_BASE64 = '...'

## 2 Definitions and arithmetic checks

Dirichlet feature smoothing uses
$$p_\mu(t\mid d)=\frac{c(t,d)+\mu p(t\mid C)}{L(d)+\mu},\quad p(t\mid C)=\frac{\sum_d c(t,d)}{\sum_d L(d)}.$$
Here L sums unigram-bigram feature occurrences. This is not a conditional bigram model and no perplexity is reported. The baseline uses mu=0. All configurations share the same training vocabulary rules.

Pearson correlation uses the sum of cross-deviations divided by the square root of the product of the two sums of squared deviations. EDA descriptors are character count, token count, unique-token count, mean token length, unique-token ratio, and the proportion of tokens of length at least seven. They are not model inputs. Correlation excludes the label and uses training documents only.

RF uses Gini impurity $1-\sum_k p_k^2$. Macro F1 averages class-level F1. Exact McNemar compares paired errors; paired bootstrap F1 resamples fixed test predictions. Examples below are arithmetic illustrations, never corpus rows.

In [ ]:
import numpy as np
c=np.array([[2.,2.],[0.,4.]])
bg=c.sum(axis=0)/c.sum();prob=(c+4*bg)/(c.sum(axis=1,keepdims=True)+4)
assert np.allclose(prob,[[.375,.625],[.125,.875]])
assert np.allclose(prob.sum(axis=1),1)
assert np.isclose(np.corrcoef([1,2,3],[2,4,6])[0,1],1)
print('Illustrative Dirichlet probabilities:',prob)
print('Illustrative parent Gini:',1-.6**2-.4**2)


## 3 Frozen setup and complete computation

Train/test split: stratified 80:20, seed 42. Train has 235 documents; test has 59 (32 negative, 27 positive). Preprocessing lowercases, strips tags, URLs, mentions and nonalphabetic characters, then collapses whitespace. No stemming or stopword removal is applied. Cleaned duplicates cannot cross the split.

S1: 100-tree balanced RF, depth None, split 2, leaf 1, sqrt feature sampling, mu=0. S2: identical RF, mu=10. S3: selected parameters from three-fold stratified training CV. CountVectorizer uses unigram-bigram features, a 12,000-feature cap and a fold-local background. Full train yields 2,625 features. Validation/test never fit the vocabulary.

Adapted search: six members, four iterations, seed 42, exact S2 incumbent, caching of duplicate decoded configurations. Search space: trees 50-180; depth None/10/20/30; split 2-10; leaf 1-4; log10(mu) from -1 to 3; max_features sqrt/0.1/0.3/0.5/0.8. Fitness is mean macro F1 on train CV. The selected fold score is not an independent test score.

The update code is an explicit adaptation. It is not presented as equation-for-equation GVOA. No random-search control, external corpus or repeated outer split is silently added. Expected runtime depends on the Colab CPU. This computation writes all CSV evidence and records actual package versions and timing.

In [ ]:
# Define corpus preprocessing and text-classification experiments.
"""Anonymous manuscript replication.
RF with Dirichlet smoothing and adapted GVOA. Performance follows the computed predictions.
"""
from pathlib import Path
import re,io,json,hashlib,math,platform,time,base64,urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import scipy,sklearn
from scipy.stats import binomtest
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,confusion_matrix,f1_score

OUT=Path('nlp_results');OUT.mkdir(exist_ok=True)
SEED=42; FOLDS=3; POP=6; ITER=4
COMMIT='1c8d2c99f281cf51e660378a06cad1fd3f1e371d'
URL=f'https://raw.githubusercontent.com/rizalespe/Dataset-Sentimen-Analisis-Bahasa-Indonesia/{COMMIT}/dataset_tweet_sentiment_cellular_service_provider.csv'
EXPECTED_SHA='6b0851bb960b2bc09bf1e92d3b735fdd2f7e19ac0f1d54c43829018095b188d6'
LOCAL_CSV=''
# The notebook supplies the verified original public corpus snapshot.
# Preserve source documents and sentiment labels.

def load_data():
    if LOCAL_CSV:
        raw=Path(LOCAL_CSV).read_bytes();route='local_snapshot'
    elif globals().get('ORIGINAL_CORPUS_BASE64'):
        raw=base64.b64decode(ORIGINAL_CORPUS_BASE64);route='embedded_verified_public_snapshot'
    else:
        raw=urllib.request.urlopen(URL,timeout=30).read();route='pinned_public_url'
    sha=hashlib.sha256(raw).hexdigest()
    if sha!=EXPECTED_SHA:raise ValueError('Corpus checksum mismatch; verify the source before continuing.')
    (OUT/'source_corpus.csv').write_bytes(raw)
    df=pd.read_csv(io.BytesIO(raw)); required={'Id','Sentiment','Text Tweet'}
    if not required.issubset(df):raise ValueError('Dataset columns do not match the expected source schema.')
    log={'raw_rows':len(df),'route':route,'source_url':URL,'commit':COMMIT,'sha256':sha}
    df=df.rename(columns={'Id':'source_id','Sentiment':'label','Text Tweet':'original_text'})
    log['missing_rows']=int(df[['original_text','label']].isna().any(axis=1).sum());df=df.dropna(subset=['original_text','label']).copy()
    df.label=df.label.astype(str).str.lower().str.strip();invalid=~df.label.isin(['positive','negative'])
    if invalid.any():raise ValueError('Unexpected sentiment label; verify corpus labels before continuing.')
    df.label=df.label.map({'negative':0,'positive':1})
    def clean(s):
        s=re.sub(r'<[^>]+>|https?://\S+|@\w+',' ',str(s).lower())
        return ' '.join(re.sub(r'[^a-z\s]',' ',s).split())
    df['text']=df.original_text.map(clean);empty=df.text.str.len()<=2;log['short_rows']=int(empty.sum());df=df[~empty]
    conflict=df.groupby('text').label.nunique();keys=conflict[conflict>1].index
    log['conflicting_groups']=len(keys);log['conflicting_rows']=int(df.text.isin(keys).sum());df=df[~df.text.isin(keys)]
    log['duplicate_rows']=int(df.duplicated('text').sum());df=df.drop_duplicates('text').reset_index(drop=True)
    log['clean_rows']=len(df);log['class_counts']={str(k):int(v) for k,v in df.label.value_counts().items()}
    df.to_csv(OUT/'clean_corpus.csv',index=False)
    return df,log

def vectorize(train,test):
    # All models share unigram-bigram features without IDF weighting.
    v=CountVectorizer(ngram_range=(1,2),max_features=12000,dtype=np.float64)
    a=v.fit_transform(train).toarray();b=v.transform(test).toarray()
    bg=a.sum(axis=0)/a.sum()
    return a,b,bg,v

def smooth_counts(counts,bg,mu):
    lengths=counts.sum(axis=1,keepdims=True)
    if mu==0:
        # Documents with no vocabulary matches receive a zero baseline vector.
        return counts/np.maximum(lengths,1)
    return (counts+mu*bg)/(lengths+mu)

BASE=dict(n_estimators=100,max_depth=None,min_samples_split=2,min_samples_leaf=1,max_features='sqrt')
DEPTHS=[None,10,20,30];FEATURES=['sqrt',.1,.3,.5,.8]
LO=np.array([50,0,2,1,-1,0.]);HI=np.array([180,3,10,4,3,4.])
def decode(z):
    z=np.clip(z,LO,HI)
    p=dict(n_estimators=int(round(z[0])),max_depth=DEPTHS[int(round(z[1]))],min_samples_split=int(round(z[2])),min_samples_leaf=int(round(z[3])),max_features=FEATURES[int(round(z[5]))])
    return p,float(10**z[4])
def rf(p):return RandomForestClassifier(**p,class_weight='balanced',random_state=SEED,n_jobs=1)

def optimize(fold_data):
    rng=np.random.default_rng(SEED);cache={};trace=[]
    def score(z):
        p,mu=decode(z);key=json.dumps([p,round(mu,10)],sort_keys=True)
        if key not in cache:
            scores=[]
            for a,b,bg,ya,yb in fold_data:
                model=rf(p).fit(smooth_counts(a,bg,mu),ya)
                scores.append(f1_score(yb,model.predict(smooth_counts(b,bg,mu)),average='macro',labels=[0,1],zero_division=0))
            cache[key]=(float(np.mean(scores)),scores,p,mu)
        return cache[key][0]
    pop=rng.uniform(LO,HI,(POP,len(LO)))
    # Include the exact S2 forest settings and smoothing strength.
    pop[0]=np.array([100,0,2,1,1,0.])
    fitness=np.array([score(z) for z in pop]);incumbent=float(fitness[0])
    for t in range(1,ITER+1):
        order=np.argsort(-fitness);pop,fitness=pop[order],fitness[order];a,b=pop[:2].copy()
        hunger=rng.random()*rng.uniform(-1,1)+2*(1-t/ITER);g=math.exp(-.4*t/ITER)
        for i in range(2,POP):
            x=pop[i].copy()
            if abs(hunger)>=1:
                if rng.random()<.5:z=x+g*rng.random(len(x))*(HI-LO)
                else:
                    beta=1.5;sig=(math.gamma(1+beta)*math.sin(math.pi*beta/2)/(math.gamma((1+beta)/2)*beta*2**((beta-1)/2)))**(1/beta)
                    levy=.01*rng.normal(0,sig,len(x))/(abs(rng.normal(size=len(x)))**(1/beta)+1e-9)
                    z=x+g*levy*(a-x)
            elif abs(hunger)>=.5:
                leader=a if rng.random()<.5 else b;z=leader+g*rng.random(len(x))*(leader-x)
            else:z=x-g*rng.random(len(x))*(x-a)+g*rng.random(len(x))*(x-b)
            z=np.clip(z,LO,HI);f=score(z)
            if f>=fitness[i]:pop[i],fitness[i]=z,f
        trace.append({'iteration':t,'best_cv_f1':float(fitness.max()),'evaluated_unique':len(cache)})
        print('Iteration',t,'CV',fitness.max(),flush=True)
    best=pop[np.argmax(fitness)];p,mu=decode(best)
    assert fitness.max()+1e-12>=incumbent
    pd.DataFrame(trace).to_csv(OUT/'search_convergence.csv',index=False)
    pd.DataFrame([{'cv_mean':v[0],'fold_scores':json.dumps(v[1]),'params':json.dumps(v[2]),'mu':v[3]} for v in cache.values()]).to_csv(OUT/'cv_candidates.csv',index=False)
    return p,mu,{'incumbent_cv':incumbent,'best_cv':float(fitness.max()),'unique_candidates':len(cache),'population':POP,'iterations':ITER}

def main():
    started=time.perf_counter();df,validation_log=load_data()
    train,test=train_test_split(np.arange(len(df)),test_size=.2,stratify=df.label,random_state=SEED)
    x=df.text.iloc[train].to_numpy();y=df.label.iloc[train].to_numpy();xt=df.text.iloc[test].to_numpy();yt=df.label.iloc[test].to_numpy()
    assert not set(x)&set(xt)
    df['split']='train';df.loc[test,'split']='test';df.to_csv(OUT/'corpus_split.csv',index=False)
    folds=list(StratifiedKFold(FOLDS,shuffle=True,random_state=SEED).split(x,y));fold_data=[]
    for tr,va in folds:
        a,b,bg,_=vectorize(x[tr],x[va]);fold_data.append((a,b,bg,y[tr],y[va]))
    bestp,bestmu,searchlog=optimize(fold_data)
    # Freeze the three configurations before evaluating test predictions.
    models=[('1 RF baseline',BASE,0.),('2 Dirichlet + RF',BASE,10.),('3 Dirichlet + RF + adapted GVOA',bestp,bestmu)]
    a,b,bg,vocab=vectorize(x,xt);rows=[];predictions={};perclass=[]
    for name,p,mu in models:
        scores=[]
        for ca,cb,cbg,ya,yb in fold_data:
            clf=rf(p).fit(smooth_counts(ca,cbg,mu),ya)
            scores.append(f1_score(yb,clf.predict(smooth_counts(cb,cbg,mu)),labels=[0,1],average='macro',zero_division=0))
        model=rf(p).fit(smooth_counts(a,bg,mu),y);pred=model.predict(smooth_counts(b,bg,mu));predictions[name]=pred
        pr,rc,f,_=precision_recall_fscore_support(yt,pred,average='macro',labels=[0,1],zero_division=0)
        cmat=confusion_matrix(yt,pred,labels=[0,1]);tn,fp,fn,tp=map(int,cmat.ravel())
        assert abs(accuracy_score(yt,pred)-(tn+tp)/len(yt))<1e-12
        rows.append(dict(model=name,features=len(vocab.vocabulary_),mu=mu,accuracy=accuracy_score(yt,pred),precision_macro=pr,recall_macro=rc,f1_macro=f,cv_mean=np.mean(scores),cv_sd=np.std(scores,ddof=1),TN=tn,FP=fp,FN=fn,TP=tp,error=fp+fn,params=json.dumps(p)))
        cp,cr,cf,support=precision_recall_fscore_support(yt,pred,labels=[0,1],zero_division=0)
        for i in range(2):perclass.append(dict(model=name,label=i,precision=cp[i],recall=cr[i],f1=cf[i],support=int(support[i])))
    metrics=pd.DataFrame(rows);metrics.to_csv(OUT/'classification_metrics.csv',index=False);pd.DataFrame(perclass).to_csv(OUT/'per_class_metrics.csv',index=False)
    preds=pd.DataFrame({'source_id':df.source_id.iloc[test].to_numpy(),'actual':yt,**predictions});preds.to_csv(OUT/'test_predictions.csv',index=False)
    pairs=[];rng=np.random.default_rng(SEED)
    names=list(predictions)
    for ia,ib in [(0,1),(1,2),(0,2)]:
        pa,pb=predictions[names[ia]],predictions[names[ib]];wrong_a=pa!=yt;wrong_b=pb!=yt
        awbr=int(np.sum(wrong_a&~wrong_b));ar_b_w=int(np.sum(~wrong_a&wrong_b))
        discordant=awbr+ar_b_w
        pval=binomtest(awbr,discordant,.5).pvalue if discordant else 1.
        delta=metrics.iloc[ib].f1_macro-metrics.iloc[ia].f1_macro
        bootstrap=[]
        for k in range(500):
            ids=rng.integers(0,len(yt),len(yt))
            bootstrap.append(f1_score(yt[ids],pb[ids],labels=[0,1],average='macro',zero_division=0)-f1_score(yt[ids],pa[ids],labels=[0,1],average='macro',zero_division=0))
        lo,hi=np.quantile(bootstrap,[.025,.975])
        pairs.append(dict(reference=names[ia],comparison=names[ib],disagreements=int(np.sum(pa!=pb)),a_wrong_b_right=awbr,a_right_b_wrong=ar_b_w,delta_f1=delta,delta_pp=100*delta,relative_percent=100*delta/metrics.iloc[ia].f1_macro,mcnemar_exact_p=float(pval),bootstrap_low=float(lo),bootstrap_high=float(hi)))
    pd.DataFrame(pairs).to_csv(OUT/'paired_comparison.csv',index=False)
    # Check probability normalization and confusion-matrix arithmetic.
    assert np.allclose(smooth_counts(a,bg,10).sum(axis=1),1.)
    assert np.all(smooth_counts(b,bg,10)>0.)
    assert all(metrics.TN+metrics.FP+metrics.FN+metrics.TP==len(test))
    assert np.allclose((metrics.TN+metrics.TP)/len(test),metrics.accuracy)
    validation_log.update(train=len(train),test=len(test),seed=SEED,test_class_counts={str(k):int(v) for k,v in Counter(yt).items()},vocabulary_size=len(vocab.vocabulary_),ngram_range=[1,2],frozen_representation='relative frequency; Dirichlet probabilities, no TF-IDF',best_mu=bestmu,best_params=bestp,search=searchlog,python=platform.python_version(),sklearn=sklearn.__version__,numpy=np.__version__,pandas=pd.__version__,scipy=scipy.__version__,seconds=time.perf_counter()-started,optimizer_status='adapted GVOA; canonical equivalence remains unverified',evaluation_status='exploratory; this split was inspected during previous development')
    (OUT/'manifest.json').write_text(json.dumps(validation_log,indent=2))
    print(metrics.to_string(index=False));print(pd.DataFrame(pairs).to_string(index=False));print(json.dumps(validation_log,indent=2))
    return metrics


metrics=main()
"""Training-text exploration without changing classifier features or settings."""
import json,re,io,base64
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

def run_eda(df,validation_log,out,show=True):
    from sklearn.model_selection import train_test_split
    out=Path(out);out.mkdir(exist_ok=True)
    tr,te=train_test_split(np.arange(len(df)),test_size=.2,stratify=df.label,random_state=42)
    train=df.iloc[tr].copy()
    def descriptors(s):
        words=s.split();lengths=[len(w) for w in words]
        return [len(s),len(words),len(set(words)),np.mean(lengths),len(set(words))/len(words),sum(len(w)>=7 for w in words)/len(words)]
    cols=['Characters','Tokens','Unique tokens','Mean token length','Unique-token ratio','Long-token ratio']
    desc=pd.DataFrame(train.text.map(descriptors).tolist(),columns=cols,index=train.index)
    desc.to_csv(out/'training_descriptors.csv',index_label='clean_row')
    summary=desc.describe().T;summary.to_csv(out/'training_statistics.csv')
    corr=desc.corr(method='pearson');corr.to_csv(out/'training_pearson_correlations.csv')
    assert np.isfinite(corr.to_numpy()).all()
    pairs=[(cols[i],cols[j],float(corr.iloc[i,j])) for i in range(len(cols)) for j in range(i)]
    strongest=max(pairs,key=lambda z:abs(z[2]))
    freq=Counter(w for s in train.text for w in s.split());top=freq.most_common(10)
    pd.DataFrame(top,columns=['token','frequency']).to_csv(out/'training_token_frequencies.csv',index=False)
    fig,axs=plt.subplots(1,3,figsize=(11,2.8))
    counts=df.label.value_counts().reindex([0,1]);axs[0].bar(['Negative','Positive'],counts,color=['#527898','#8cae91'])
    for i,n in enumerate(counts):axs[0].text(i,n+2,str(n),ha='center',fontsize=9)
    axs[0].set_ylim(0,190);axs[0].set_title('Labeled corpus class distribution');axs[0].set_ylabel('Documents')
    for label,col,name in [(0,'#527898','Negative'),(1,'#8cae91','Positive')]:
        axs[1].hist(desc.loc[train.label==label,'Tokens'],bins=np.arange(0,int(np.ceil(desc.Tokens.max()/5)*5)+6,5),alpha=.6,color=col,label=name)
    axs[1].set_title('Training-text length');axs[1].set_xlabel('Token count');axs[1].legend(fontsize=8)
    axs[2].barh([w for w,n in top][::-1],[n for w,n in top][::-1],color='#527898');axs[2].set_title('Ten most frequent training tokens');axs[2].set_xlabel('Occurrences')
    fig.tight_layout();fig.savefig(out/'eda_train.png',dpi=180,bbox_inches='tight')
    if show:plt.show()
    plt.close(fig)
    fig,ax=plt.subplots(figsize=(7,4.5));im=ax.imshow(corr,vmin=-1,vmax=1,cmap='RdBu_r')
    labels=['Characters','Tokens','Unique tokens','Mean length','Unique ratio','Long ratio']
    ax.set_xticks(range(6),labels,rotation=30,ha='right',fontsize=8);ax.set_yticks(range(6),labels,fontsize=8)
    for i in range(6):
        for j in range(6):ax.text(j,i,f'{corr.iloc[i,j]:.2f}',ha='center',va='center',fontsize=9,color='white' if abs(corr.iloc[i,j])>.65 else 'black')
    fig.colorbar(im,ax=ax,label='Pearson r');ax.set_title('Text-descriptor correlations across 235 training documents');fig.tight_layout();fig.savefig(out/'training_correlation_heatmap.png',dpi=180,bbox_inches='tight')
    if show:plt.show()
    plt.close(fig)
    q1,q3=desc.Tokens.quantile([.25,.75]);upper=q3+1.5*(q3-q1);outliers=int((desc.Tokens>upper).sum())
    info=dict(train=len(tr),test=len(te),strongest_pair=strongest,word_count_median=float(desc.Tokens.median()),word_count_min=int(desc.Tokens.min()),word_count_max=int(desc.Tokens.max()),word_count_mean=float(desc.Tokens.mean()),iqr_upper=float(upper),long_text_flags=outliers,top_word=top[0],correlation_scope='train only; descriptors not RF predictors')
    (out/'eda_manifest.json').write_text(json.dumps(info,indent=2,ensure_ascii=False))
    print(summary.round(3).to_string());print(json.dumps(info,indent=2,ensure_ascii=False))
    return info,summary,corr


eda_df,eda_validation_log=load_data()
eda_info,eda_summary,eda_corr=run_eda(eda_df,eda_validation_log,OUT)
assert np.isclose(np.corrcoef([1,2,3],[2,4,6])[0,1],1.)


## 4 Publication figures and metric reconstruction

The figures and tables below read the current run. S1, S2 and S3 follow the configuration order above. Confusion-matrix rows are actual classes; columns are predictions, negative then positive. Per-class metrics distinguish sentiment-positive from generic correctness. Assertions reconstruct CM, accuracy, precision, recall and F1 from individual predictions. No assertion forces an improvement or a model ranking.

In [ ]:
from pathlib import Path
import re,json,base64,io,hashlib
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix,precision_recall_fscore_support,accuracy_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT=Path('nlp_results/figures');OUT.mkdir(exist_ok=True);R=Path('nlp_results')
metrics=pd.read_csv(R/'classification_metrics.csv');data=pd.read_csv(R/'corpus_split.csv');stats=pd.read_csv(R/'training_descriptors.csv');corr=pd.read_csv(R/'training_pearson_correlations.csv',index_col=0)
plt.rcParams.update({'font.family':'DejaVu Serif','font.size':7,'axes.titlesize':8,'axes.labelsize':7,'xtick.labelsize':7,'ytick.labelsize':7,'legend.fontsize':7})
fig,axes=plt.subplots(2,1,figsize=(2.85,2.75));counts=data.label.value_counts().reindex([0,1]);axes[0].bar(['Negative','Positive'],counts,color=['#51758c','#adadad'])
for i,n in enumerate(counts):axes[0].text(i,n+2,str(n),ha='center')
axes[0].set_ylim(0,190);axes[0].set_ylabel('Documents');axes[0].set_title('(a) Labeled corpus')
axes[1].hist(stats.Tokens,bins=np.arange(0,26,3),color='#51758c',edgecolor='white');axes[1].set_xlabel('Tokens per document');axes[1].set_ylabel('Training documents');axes[1].set_title('(b) Training text length')
fig.tight_layout();fig.savefig(OUT/'figure1.png',dpi=400);plt.close(fig)
fig,ax=plt.subplots(figsize=(3,2.75));img=ax.imshow(corr,vmin=-1,vmax=1,cmap='RdBu_r');labels=['Characters','Tokens','Unique tokens','Mean length','Unique ratio','Long ratio'];ax.set_xticks(range(6),labels,rotation=42,ha='right');ax.set_yticks(range(6),labels)
for i in range(6):
 for j in range(6):ax.text(j,i,f'{corr.iloc[i,j]:.2f}',ha='center',va='center',fontsize=6,color='white' if abs(corr.iloc[i,j])>.65 else 'black')
fig.colorbar(img,ax=ax,fraction=.04,pad=.04);fig.tight_layout();fig.savefig(OUT/'figure2.png',dpi=400);plt.close(fig)
fig,axes=plt.subplots(1,3,figsize=(5.7,1.85))
for i,(ax,(_,r)) in enumerate(zip(axes,metrics.iterrows())):
 cm=np.array([[r.TN,r.FP],[r.FN,r.TP]],int);ax.imshow(cm,vmin=0,vmax=32,cmap='Blues');ax.set_title(['S1 Relative RF','S2 Fixed smoothing','S3 Adapted search'][i]);ax.set_xticks([0,1],['Negative','Positive']);ax.set_yticks([0,1],['Negative','Positive']);ax.set_xlabel('Predicted');ax.set_ylabel('Actual')
 for u in range(2):
  for v in range(2):ax.text(v,u,str(cm[u,v]),ha='center',va='center',fontsize=8,color='white' if cm[u,v]>16 else 'black')
fig.tight_layout();fig.savefig(OUT/'figure3.png',dpi=400);plt.close(fig)

from IPython.display import display,Image
for figure in ['figure1.png','figure2.png','figure3.png']:
    display(Image(filename=str(OUT/figure)))
pred=pd.read_csv(R/'test_predictions.csv')
for _,row in metrics.iterrows():
    truth=pred.actual;prediction=pred[row.model]
    cm=confusion_matrix(truth,prediction,labels=[0,1])
    pr,rc,f,_=precision_recall_fscore_support(truth,prediction,labels=[0,1],average='macro',zero_division=0)
    assert np.array_equal(cm.ravel(),[row.TN,row.FP,row.FN,row.TP])
    assert np.allclose([accuracy_score(truth,prediction),pr,rc,f],[row.accuracy,row.precision_macro,row.recall_macro,row.f1_macro])
display(metrics[['model','accuracy','precision_macro','recall_macro','f1_macro','cv_mean','error']])
display(pd.read_csv(R/'per_class_metrics.csv'))
display(pd.read_csv(R/'paired_comparison.csv'))
print('PASS: all metrics reconstructed from individual predictions.')


## 5 Conclusion matrix and interpretation

| Question | Observed macro F1 | Conclusion |
|---|---|---|
| Does fixed smoothing improve the baseline? | 0.8055 to 0.5743 | No on this split. |
| Does adapted search improve fixed smoothing? | 0.5743 to 0.7065 | Yes, +0.1322. |
| Does the complete pipeline beat the baseline? | 0.7065 below 0.8055 | No. |

The test split was used during earlier development. Treat all scores and p-values as exploratory. Joint changes to mu and RF settings prevent attribution of the recovery to the update rule alone. Bootstrap intervals condition on fixed predictions; the 500 resamples do not measure training variance. McNemar p=0.0703 versus the baseline does not prove equivalence. Label reliability, canonical optimizer equivalence, external validity and equal-budget optimization comparisons remain open.

This author-identified manuscript follows the requested author order. JIKI requests a separate anonymous version for first submission. The notebook supplies EDA, heatmap, setup, CM, metrics, individual predictions and raw-source provenance. Formatting compliance does not establish scientific acceptance. AI assistance supported drafting and code review; reported numbers originate from executed code.

## Recent NLP literature and research scope

This targeted literature update uses publisher and ACL Anthology records accessed on November 17, 2026. The 25 recent papers date from 2022-2026, which falls within the preceding five years. Seven earlier works preserve attribution of foundational methods, evaluation procedures, and the original corpus. This is a focused narrative synthesis, not an exhaustive systematic review. Performance values from different corpora are not pooled.

Recent reviews place RF within a broader text-classification comparison. Palanivinayagam, El-Bayeh, and Damaševičius [6] compare studies across representations, training procedures, evaluation metrics, and data partitions. Tiwari and colleagues [7] compare eight ensemble learners on four sentiment datasets. These studies support retaining an RF baseline and controlling the representation. Their results do not establish which smoothing strength will work on this Indonesian corpus.

Xu, Chang, and Jayne [8] identify methodological and dataset challenges in social-media sentiment analysis. Mao, Liu, and Zhang [9] review sentiment methods and their application-dependent limitations. Accordingly, we interpret a score within its corpus, labeling scheme, and evaluation protocol. A higher score from a different review domain cannot serve as a direct improvement over the present telecommunications baseline.

Recent smoothing studies modify different parts of the text-classification pipeline. Wu and colleagues [10] replace one-hot token representations with masked-language-model distributions for augmentation. Liu and colleagues [11] derive soft training targets from keywords, while Ren and colleagues [12] learn discrimination-aware label distributions. Our Dirichlet transformation instead changes document-feature probabilities while retaining hard sentiment labels. These interventions require separate definitions and baselines; evidence for one does not establish an RF benefit for another.

Yang and colleagues [13] examine label smoothing for adversarial robustness in pretrained text models. Sultan [14] finds that label smoothing and knowledge distillation can alter classification confidence in different directions. Lu and colleagues [15] formulate adaptive label regularization and evaluate its NLP component on machine translation. Together, these studies motivate specifying the smoothing target, objective, and evaluation condition. They do not directly validate collection-prior smoothing of RF inputs.

For Indonesian evaluation, NusaX [16] provides parallel sentiment resources for ten local languages, whereas NusaWrites [17] studies corpus construction and classification across underrepresented languages. Both offer relevant directions for external evaluation. Neither resource replaces the telecommunications corpus used here. A future comparison must first align language, sentiment categories, document units, and reuse conditions, then fix its training and testing protocol.

NLP reproducibility research separates rerunning code from establishing a stable model comparison. Belz, Popović, and Mille [18] quantify differences between reproduced results. Magnusson, Smith, and Dodge [19] examine reporting of validation results, hyperparameters, efficiency, and code availability. These findings motivate exporting the corpus identity, model settings, fold scores, and document-level predictions in the notebook. Such records support reproduction of the reported comparison without establishing external validity.

Xue and colleagues [20] study how corpus-splitting estimators affect reproducibility of NLP model comparisons. Popović and colleagues [21] examine variation across seeds, runtime environments, and dependency versions in text simplification. Their findings support the planned repeated evaluation and complete reporting of seeds. They also explain why the present reused split cannot justify a general performance ranking.

Mu, Jin, Song, and Aletras [22] identify duplication, label inconsistencies, and leakage across social-media datasets. Their analysis supports checking document overlap before text classification. Here, exact cleaned-text deduplication addresses identical inputs; it does not establish the absence of paraphrases or near-duplicate messages. We therefore retain a separate requirement for near-duplicate screening in future corpus evaluation.

Linguistic annotation also requires contextual interpretation. Sandri and colleagues [23] distinguish reasons for annotator disagreement in subjective language tasks. EPIC [24] and MultiPICo [25] examine perspectives in irony annotation across language varieties and annotator groups. These studies support recording disagreement and context during error analysis. Disagreement alone does not justify changing a corpus label to match a classifier prediction.

Jiang and colleagues [26] address implicit sentiment and polarity shifts through semantic simplification. Xu and colleagues [27] study label-dependent noise in multi-label text classification. These studies concern different tasks and models, but they motivate distinguishing linguistic failure from uncertain annotation in the proposed error analysis. We have not implemented semantic simplification, estimated a label-noise model, or measured their effects on this binary corpus.

Huang and colleagues [28] rerank difficult text-classification predictions using label semantics. Their pretrained-model setting offers a candidate semantic comparison for future work, subject to training-only selection and a separate evaluation. We do not treat that method as an already tested RF component. The next comparison should include a linear text classifier and a contextual text classifier alongside RF before attributing any improvement to smoothing or search.

In [ ]:
# Check the reference list used by the English manuscript and notebook.
import json
REFERENCE_RECORDS = json.loads(r'''[{"id": 1, "title": "Random Forests", "year": 2001, "doi": "10.1023/A:1010933404324", "url": "https://doi.org/10.1023/A:1010933404324"}, {"id": 2, "title": "A study of smoothing methods for language models applied to information retrieval", "year": 2004, "doi": "10.1145/984321.984322", "url": "https://doi.org/10.1145/984321.984322"}, {"id": 3, "title": "Griffon vultures optimization algorithm for solving optimization problems", "year": 2025, "doi": "10.1016/j.eswa.2025.127206", "url": "https://doi.org/10.1016/j.eswa.2025.127206"}, {"id": 4, "title": "Sentiment Analysis of User Satisfaction with Indonesian Cellular Telecommunications Providers on Twitter Using Support Vector Machines and Lexicon-Based Features [English translation]", "year": 2017, "doi": "", "url": "https://j-ptiik.ub.ac.id/index.php/j-ptiik/article/view/628"}, {"id": 5, "title": "Optimizing Online Gambling Site Detection via XLM-RoBERTa and ResNet34-Based Early Fusion", "year": 2026, "doi": "", "url": "https://jiki.cs.ui.ac.id/index.php/jiki/article/view/1848"}, {"id": 6, "title": "Twenty Years of Machine-Learning-Based Text Classification: A Systematic Review", "year": 2023, "doi": "10.3390/a16050236", "url": "https://doi.org/10.3390/a16050236"}, {"id": 7, "title": "A systematic review of social network sentiment analysis with comparative study of ensemble-based techniques", "year": 2023, "doi": "10.1007/s10462-023-10472-w", "url": "https://doi.org/10.1007/s10462-023-10472-w"}, {"id": 8, "title": "A systematic review of social media-based sentiment analysis: Emerging trends and challenges", "year": 2022, "doi": "10.1016/j.dajour.2022.100073", "url": "https://doi.org/10.1016/j.dajour.2022.100073"}, {"id": 9, "title": "Sentiment analysis methods, applications, and challenges: A systematic literature review", "year": 2024, "doi": "10.1016/j.jksuci.2024.102048", "url": "https://doi.org/10.1016/j.jksuci.2024.102048"}, {"id": 10, "title": "Text Smoothing: Enhance Various Data Augmentation Methods on Text Classification Tasks", "year": 2022, "doi": "10.18653/v1/2022.acl-short.97", "url": "https://aclanthology.org/2022.acl-short.97/"}, {"id": 11, "title": "Label Smoothing for Text Mining", "year": 2022, "doi": "", "url": "https://aclanthology.org/2022.coling-1.193/"}, {"id": 12, "title": "Learning label smoothing for text classification", "year": 2024, "doi": "10.7717/peerj-cs.2005", "url": "https://doi.org/10.7717/peerj-cs.2005"}, {"id": 13, "title": "In and Out-of-Domain Text Adversarial Robustness via Label Smoothing", "year": 2023, "doi": "10.18653/v1/2023.acl-short.58", "url": "https://aclanthology.org/2023.acl-short.58/"}, {"id": 14, "title": "Knowledge Distillation ≈ Label Smoothing: Fact or Fallacy?", "year": 2023, "doi": "10.18653/v1/2023.emnlp-main.271", "url": "https://aclanthology.org/2023.emnlp-main.271/"}, {"id": 15, "title": "LABO: Towards Learning Optimal Label Regularization via Bi-level Optimization", "year": 2023, "doi": "10.18653/v1/2023.findings-acl.356", "url": "https://aclanthology.org/2023.findings-acl.356/"}, {"id": 16, "title": "NusaX: Multilingual Parallel Sentiment Dataset for 10 Indonesian Local Languages", "year": 2023, "doi": "10.18653/v1/2023.eacl-main.57", "url": "https://aclanthology.org/2023.eacl-main.57/"}, {"id": 17, "title": "NusaWrites: Constructing High-Quality Corpora for Underrepresented and Extremely Low-Resource Languages", "year": 2023, "doi": "10.18653/v1/2023.ijcnlp-main.60", "url": "https://aclanthology.org/2023.ijcnlp-main.60/"}, {"id": 18, "title": "Quantified Reproducibility Assessment of NLP Results", "year": 2022, "doi": "10.18653/v1/2022.acl-long.2", "url": "https://aclanthology.org/2022.acl-long.2/"}, {"id": 19, "title": "Reproducibility in NLP: What Have We Learned from the Checklist?", "year": 2023, "doi": "10.18653/v1/2023.findings-acl.809", "url": "https://aclanthology.org/2023.findings-acl.809/"}, {"id": 20, "title": "We Need to Talk About Reproducibility in NLP Model Comparison", "year": 2023, "doi": "10.18653/v1/2023.emnlp-main.586", "url": "https://aclanthology.org/2023.emnlp-main.586/"}, {"id": 21, "title": "Exploring Variation of Results from Different Experimental Conditions", "year": 2023, "doi": "10.18653/v1/2023.findings-acl.172", "url": "https://aclanthology.org/2023.findings-acl.172/"}, {"id": 22, "title": "Enhancing Data Quality through Simple De-duplication: Navigating Responsible Computational Social Science Research", "year": 2024, "doi": "10.18653/v1/2024.emnlp-main.694", "url": "https://aclanthology.org/2024.emnlp-main.694/"}, {"id": 23, "title": "Why Don’t You Do It Right? Analysing Annotators’ Disagreement in Subjective Tasks", "year": 2023, "doi": "10.18653/v1/2023.eacl-main.178", "url": "https://aclanthology.org/2023.eacl-main.178/"}, {"id": 24, "title": "EPIC: Multi-Perspective Annotation of a Corpus of Irony", "year": 2023, "doi": "10.18653/v1/2023.acl-long.774", "url": "https://aclanthology.org/2023.acl-long.774/"}, {"id": 25, "title": "MultiPICo: Multilingual Perspectivist Irony Corpus", "year": 2024, "doi": "10.18653/v1/2024.acl-long.849", "url": "https://aclanthology.org/2024.acl-long.849/"}, {"id": 26, "title": "Semantic Simplification for Sentiment Classification", "year": 2022, "doi": "10.18653/v1/2022.emnlp-main.757", "url": "https://aclanthology.org/2022.emnlp-main.757/"}, {"id": 27, "title": "Enhancing Multi-Label Text Classification under Label-Dependent Noise: A Label-Specific Denoising Framework", "year": 2024, "doi": "10.18653/v1/2024.findings-emnlp.324", "url": "https://aclanthology.org/2024.findings-emnlp.324/"}, {"id": 28, "title": "Logits Reranking via Semantic Labels for Hard Samples in Text Classification", "year": 2024, "doi": "10.18653/v1/2024.findings-emnlp.657", "url": "https://aclanthology.org/2024.findings-emnlp.657/"}, {"id": 29, "title": "On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation", "year": 2010, "doi": "", "url": "https://jmlr.org/papers/v11/cawley10a.html"}, {"id": 30, "title": "Bias in error estimation when using cross-validation for model selection", "year": 2006, "doi": "10.1186/1471-2105-7-91", "url": "https://doi.org/10.1186/1471-2105-7-91"}, {"id": 31, "title": "Random Search for Hyper-Parameter Optimization", "year": 2012, "doi": "", "url": "https://jmlr.org/papers/v13/bergstra12a.html"}, {"id": 32, "title": "A quantum-inspired multimodal sentiment analysis framework", "year": 2018, "doi": "10.1016/j.tcs.2018.04.029", "url": "https://doi.org/10.1016/j.tcs.2018.04.029"}]''')
recent_records = [r for r in REFERENCE_RECORDS if 2022 <= r['year'] <= 2026]
assert len(REFERENCE_RECORDS) == 32
assert len(recent_records) >= 25
assert [r['id'] for r in REFERENCE_RECORDS] == list(range(1, 33))
dois = [r['doi'].lower() for r in REFERENCE_RECORDS if r['doi']]
assert len(dois) == len(set(dois))
assert len({r['title'].lower() for r in REFERENCE_RECORDS}) == len(REFERENCE_RECORDS)
print(f"Cited works: {len(REFERENCE_RECORDS)}; papers from 2022-2026: {len(recent_records)}")
print("Reference identifiers are contiguous and DOI records are unique.")


## References

[1] L. Breiman, "Random forests," Machine Learning, vol. 45, pp. 5-32, 2001. doi: 10.1023/A:1010933404324

[2] C. Zhai and J. Lafferty, "A study of smoothing methods for language models applied to information retrieval," ACM Trans. Inf. Syst., vol. 22, no. 2, pp. 179-214, 2004. doi: 10.1145/984321.984322

[3] D. O. Hasan, H. M. Mohammed, and Z. K. Abdul, "Griffon vultures optimization algorithm for solving optimization problems," Expert Syst. Appl., vol. 276, Art. no. 127206, 2025. doi: 10.1016/j.eswa.2025.127206.

[4] U. Rofiqoh, R. S. Perdana, and M. A. Fauzi, "Sentiment Analysis of User Satisfaction with Indonesian Cellular Telecommunications Providers on Twitter Using Support Vector Machines and Lexicon-Based Features [English translation]," J-PTIIK, vol. 1, no. 12, pp. 1725-1732, 2017. [Online]. Available: https://j-ptiik.ub.ac.id/index.php/j-ptiik/article/view/628

[5] R. Nugroho, and D. Kurniawan, "Optimizing Online Gambling Site Detection via XLM-RoBERTa and ResNet34-Based Early Fusion," J. Ilmu Komput. Inf., vol. 19, no. 2, pp. 281-292, 2026. [Online]. Available: https://jiki.cs.ui.ac.id/index.php/jiki/article/view/1848

[6] A. Palanivinayagam, C. Z. El-Bayeh, and R. Damaševičius, "Twenty Years of Machine-Learning-Based Text Classification: A Systematic Review," Algorithms, vol. 16, no. 5, Art. no. 236, 2023. doi: 10.3390/a16050236.

[7] D. Tiwari, B. Nagpal, B. S. Bhati, A. Mishra, and M. Kumar, "A systematic review of social network sentiment analysis with comparative study of ensemble-based techniques," Artif. Intell. Rev., vol. 56, no. 11, pp. 13407-13461, 2023. doi: 10.1007/s10462-023-10472-w.

[8] Q. A. Xu, V. Chang, and C. Jayne, "A systematic review of social media-based sentiment analysis: Emerging trends and challenges," Decision Anal. J., vol. 3, Art. no. 100073, 2022. doi: 10.1016/j.dajour.2022.100073.

[9] Y. Mao, Q. Liu, and Y. Zhang, "Sentiment analysis methods, applications, and challenges: A systematic literature review," J. King Saud Univ. Comput. Inf. Sci., vol. 36, no. 4, Art. no. 102048, 2024. doi: 10.1016/j.jksuci.2024.102048.

[10] X. Wu, C. Gao, M. Lin, L. Zang, and S. Hu, "Text Smoothing: Enhance Various Data Augmentation Methods on Text Classification Tasks," in Proc. 60th Annu. Meeting of the Assoc. Comput. Linguistics (Volume 2: Short Papers), 2022, pp. 871-875. doi: 10.18653/v1/2022.acl-short.97.

[11] P. Liu, X. Xi, W. Ye, and S. Zhang, "Label Smoothing for Text Mining," in Proc. 29th Int. Conf. on Computational Linguistics, 2022, pp. 2210-2219. [Online]. Available: https://aclanthology.org/2022.coling-1.193/

[12] H. Ren, Y. Zhao, Y. Zhang, and W. Sun, "Learning label smoothing for text classification," PeerJ Comput. Sci., vol. 10, Art. no. e2005, 2024. doi: 10.7717/peerj-cs.2005.

[13] Y. Yang, S. Dan, D. Roth, and I. Lee, "In and Out-of-Domain Text Adversarial Robustness via Label Smoothing," in Proc. 61st Annu. Meeting of the Assoc. Comput. Linguistics (Volume 2: Short Papers), 2023, pp. 657-669. doi: 10.18653/v1/2023.acl-short.58.

[14] M. Sultan, "Knowledge Distillation ≈ Label Smoothing: Fact or Fallacy?," in Proc. 2023 Conference on Empirical Methods in Natural Language Processing, 2023, pp. 4469-4477. doi: 10.18653/v1/2023.emnlp-main.271.

[15] P. Lu, A. Rashid, I. Kobyzev, M. Rezagholizadeh, and P. Langlais, "LABO: Towards Learning Optimal Label Regularization via Bi-level Optimization," in Findings of the Assoc. Comput. Linguistics: ACL 2023, 2023, pp. 5759-5774. doi: 10.18653/v1/2023.findings-acl.356.

[16] G. I. Winata, A. F. Aji, S. Cahyawijaya, R. Mahendra, F. Koto, A. Romadhony, K. Kurniawan, D. Moeljadi, R. E. Prasojo, P. Fung, T. Baldwin, J. H. Lau, R. Sennrich, and S. Ruder, "NusaX: Multilingual Parallel Sentiment Dataset for 10 Indonesian Local Languages," in Proc. 17th Conference of the European Chapter of the Assoc. Comput. Linguistics, 2023, pp. 815-834. doi: 10.18653/v1/2023.eacl-main.57.

[17] S. Cahyawijaya, H. Lovenia, F. Koto, D. Adhista, E. Dave, S. Oktavianti, S. Akbar, J. Lee, N. Shadieq, T. W. Cenggoro, H. Linuwih, B. Wilie, G. Muridan, G. Winata, D. Moeljadi, A. F. Aji, A. Purwarianti, and P. Fung, "NusaWrites: Constructing High-Quality Corpora for Underrepresented and Extremely Low-Resource Languages," in Proc. 13th International Joint Conference on Natural Language Processing and the 3rd Conference of the Asia-Pacific Chapter of the Assoc. Comput. Linguistics (Volume 1: Long Papers), 2023, pp. 921-945. doi: 10.18653/v1/2023.ijcnlp-main.60.

[18] A. Belz, M. Popovic, and S. Mille, "Quantified Reproducibility Assessment of NLP Results," in Proc. 60th Annu. Meeting of the Assoc. Comput. Linguistics (Volume 1: Long Papers), 2022, pp. 16-28. doi: 10.18653/v1/2022.acl-long.2.

[19] I. Magnusson, N. A. Smith, and J. Dodge, "Reproducibility in NLP: What Have We Learned from the Checklist?," in Findings of the Assoc. Comput. Linguistics: ACL 2023, 2023, pp. 12789-12811. doi: 10.18653/v1/2023.findings-acl.809.

[20] Y. Xue, X. Cao, X. Yang, Y. Wang, R. Wang, and J. Li, "We Need to Talk About Reproducibility in NLP Model Comparison," in Proc. 2023 Conference on Empirical Methods in Natural Language Processing, 2023, pp. 9424-9434. doi: 10.18653/v1/2023.emnlp-main.586.

[21] M. Popović, M. Arvan, N. Parde, and A. Belz, "Exploring Variation of Results from Different Experimental Conditions," in Findings of the Assoc. Comput. Linguistics: ACL 2023, 2023, pp. 2746-2757. doi: 10.18653/v1/2023.findings-acl.172.

[22] Y. Mu, M. Jin, X. Song, and N. Aletras, "Enhancing Data Quality through Simple De-duplication: Navigating Responsible Computational Social Science Research," in Proc. 2024 Conference on Empirical Methods in Natural Language Processing, 2024, pp. 12477-12492. doi: 10.18653/v1/2024.emnlp-main.694.

[23] M. Sandri, E. Leonardelli, S. Tonelli, and E. Jezek, "Why Don’t You Do It Right? Analysing Annotators’ Disagreement in Subjective Tasks," in Proc. 17th Conference of the European Chapter of the Assoc. Comput. Linguistics, 2023, pp. 2428-2441. doi: 10.18653/v1/2023.eacl-main.178.

[24] S. Frenda, A. Pedrani, V. Basile, S. M. Lo, A. T. Cignarella, R. Panizzon, C. Marco, B. Scarlini, V. Patti, C. Bosco, and D. Bernardi, "EPIC: Multi-Perspective Annotation of a Corpus of Irony," in Proc. 61st Annu. Meeting of the Assoc. Comput. Linguistics (Volume 1: Long Papers), 2023, pp. 13844-13857. doi: 10.18653/v1/2023.acl-long.774.

[25] S. Casola, S. Frenda, S. M. Lo, E. Sezerer, A. Uva, V. Basile, C. Bosco, A. Pedrani, C. Rubagotti, V. Patti, and D. Bernardi, "MultiPICo: Multilingual Perspectivist Irony Corpus," in Proc. 62nd Annu. Meeting of the Assoc. Comput. Linguistics (Volume 1: Long Papers), 2024, pp. 16008-16021. doi: 10.18653/v1/2024.acl-long.849.

[26] X. Jiang, Z. Wang, and G. Zhou, "Semantic Simplification for Sentiment Classification," in Proc. 2022 Conference on Empirical Methods in Natural Language Processing, 2022, pp. 11022-11032. doi: 10.18653/v1/2022.emnlp-main.757.

[27] P. Xu, L. Jing, and J. Yu, "Enhancing Multi-Label Text Classification under Label-Dependent Noise: A Label-Specific Denoising Framework," in Findings of the Assoc. Comput. Linguistics: EMNLP 2024, 2024, pp. 5674-5688. doi: 10.18653/v1/2024.findings-emnlp.324.

[28] P. Huang, J. Huang, Y. Xu, W. Li, and X. Xiao, "Logits Reranking via Semantic Labels for Hard Samples in Text Classification," in Findings of the Assoc. Comput. Linguistics: EMNLP 2024, 2024, pp. 11250-11262. doi: 10.18653/v1/2024.findings-emnlp.657.

[29] G. C. Cawley and N. L. C. Talbot, "On over-fitting in model selection and subsequent selection bias in performance evaluation," J. Mach. Learn. Res., vol. 11, pp. 2079-2107, 2010. [Online]. Available: https://jmlr.org/papers/v11/cawley10a.html

[30] S. Varma and R. Simon, "Bias in error estimation when using cross-validation for model selection," BMC Bioinformatics, vol. 7, Art. no. 91, 2006. doi: 10.1186/1471-2105-7-91

[31] J. Bergstra and Y. Bengio, "Random search for hyper-parameter optimization," J. Mach. Learn. Res., vol. 13, pp. 281-305, 2012. [Online]. Available: https://jmlr.org/papers/v13/bergstra12a.html

[32] Y. Zhang, D. Song, P. Zhang, P. Wang, J. Li, X. Li, and B. Wang, "A quantum-inspired multimodal sentiment analysis framework," Theoretical Computer Science, vol. 752, pp. 21-40, 2018, doi: 10.1016/j.tcs.2018.04.029.

## JIKI format and evidence checks

The manuscript uses the supplied Word template with A4 paper, 3 cm margins, two columns, English text, numbered sections and IEEE references. The 182-word single-paragraph abstract satisfies the template maximum of 200 and the website range of 150-250. Five keywords and an 11-word title satisfy the supplied limits. Body text uses 11 pt following the current website, while captions and tables use 8 pt and the abstract uses 10 pt. The template states 10 pt body text; the website states 11-12 pt. This choice follows the current website while retaining template geometry. The Word equations are editable Office Math, not images. Tables have no vertical rules. Authors and affiliations are included in this requested identified version; remove them from the anonymous first-submission version required by JIKI. Figures and tables occupy page-top regions; IEEE citations link to numbered references. Reference numbering follows first appearance in the manuscript. The bibliography contains 32 cited works, including 25 papers published in 2022-2026; seven older sources identify foundational methods and corpus provenance. The RIS file contains the same 32 records. DOI metadata for 27 works matched the resolver records; the other five works use publisher or journal landing pages. These checks do not establish journal quartiles.

The draft does not establish algorithmic novelty or confirmed generalization. Canonical GVOA verification, external evaluation, independent label review and equal-budget search controls remain outstanding research tasks. Human authors must complete journal declarations and author metadata separately before submission. Formatting alone does not imply journal acceptance.


## Credit Authorship Contribution Statement

Draft contribution allocation for author verification. Each author should verify the proposed roles for this NLP study.

Alex Gufron: Conceptualization, Methodology, Software, Writing - original draft. Florentina Yuni Arini: Methodology, Validation, Writing - review & editing. Nizirwan Anwar: Formal analysis, Supervision, Writing - review & editing. Alamsyah: Supervision, Validation, Investigation, Writing - review & editing.

## Research development and bounded literature gap

The development agenda concerns unresolved comparisons within the reviewed evidence, not a claim that no researcher has combined smoothing and RF. Zhang et al. [32] smooth multimodal document representations and evaluate RF and support vector machines (SVM) in a multimodal sentiment framework. Their representation differs from the present n-gram features. The studies cited here do not establish whether our observed recovery comes from smoothing, forest settings, or the adapted search rule. This targeted literature check is not an exhaustive systematic review.

RQ4 concerns mechanism attribution: how much do collection-prior addition and document-length shrinkage each contribute? A planned two-by-two ablation will enable or disable each component while fixing the vocabulary, RF settings and folds. The four transformations will use c/L, (c+mu pC)/L, c/(L+mu), and (c+mu pC)/(L+mu), for L>0. The intermediate arms are feature interventions, not normalized probability models. The notebook states a unit-denominator convention for zero-vocabulary documents in the unshrunk arms. Fixed positive mu values will allow component contrasts without simultaneous RF tuning.

RQ5 concerns optimization attribution: does adapted vulture search improve the same objective more than random search under the same budget? A planned comparison will give both searches 22 unique decoded evaluations, the same domain, training folds, seed schedule and fixed-smoothed incumbent. It will report best-so-far macro F1 against evaluation count and elapsed time. Ablations will tune mu alone, RF settings alone, and both together. Canonical GVOA will enter only after equation-level implementation checks and benchmark validation. These controls extend the present evaluation; they have not produced results in this manuscript.

RQ6 concerns transfer: does the ordering of configurations persist across domains and split seeds? A planned nested design will use five outer stratified folds, three inner folds and five prespecified seeds. Each outer training fold will fit its own vocabulary and collection prior. The used telecommunications split will remain exploratory. A second independently sourced Indonesian corpus requires provenance, label definitions and reuse-condition checks before inclusion. No second corpus or cross-domain score is claimed here. Reports will distinguish within-domain outer-fold evaluation from transfer to another corpus.

RQ7 concerns annotation and linguistic failure modes. Two independent annotators will inspect a prespecified sample without model predictions, record disagreements and adjudication, and report agreement before labels change. The analysis will distinguish negation, irony, slang, anonymization and all-out-of-vocabulary cases. Sampling and subgroup rules will precede inspection. The purpose is to test source-label reliability and error concentration, while preserving independent annotation decisions.

The planned primary outcome is paired outer-evaluation macro F1; class metrics, balanced accuracy, uncertainty and resource use are secondary. A prespecified hierarchical analysis will retain corpus and repetition structure rather than treat overlapping folds as independent samples. All evaluated configurations and seeds will remain visible. The optional notebook prototype performs only training-CV factorial screening when enabled. It does not implement the full nested, cross-corpus or optimizer-control protocol. It is disabled by default and has generated no new experimental scores for this paper.

Reference [32]: Y. Zhang, D. Song, P. Zhang, P. Wang, J. Li, X. Li, and B. Wang, "A quantum-inspired multimodal sentiment analysis framework," Theoretical Computer Science, vol. 752, pp. 21-40, 2018, doi: 10.1016/j.tcs.2018.04.029.

Targeted search on 15 November 2026 used the queries `"Dirichlet smoothing" "random forest" sentiment` and `"Griffon vultures" "sentiment" classification`. Discovery used web search; verification checked the accepted manuscript at https://oro.open.ac.uk/54814/1/54814.pdf, especially equation (9) and the RF/SVM experiment description. Source coverage is selective, not a systematic claim that all prior research lacks these controls.

In [ ]:
"""Optional training-only screening prototype, not a confirmatory experiment."""
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

def factorial_features(counts,background,mu,numerator_prior,denominator_shrink):
    counts=np.asarray(counts,dtype=float)
    length=counts.sum(axis=1,keepdims=True)
    numerator=counts+(mu*background if numerator_prior else 0.)
    denominator=length+mu if denominator_shrink else np.maximum(length,1.)
    return numerator/denominator

def run_training_screening(root='nlp_results',mus=(1.,10.),seed=42):
    """Reuse only the archived training rows; never inspect the test scores."""
    frame=pd.read_csv(Path(root)/'corpus_split.csv')
    frame=frame.loc[frame['split'].eq('train')].copy()
    texts=frame['text'].to_numpy();labels=frame['label'].to_numpy()
    splitter=StratifiedKFold(3,shuffle=True,random_state=seed)
    rows=[]
    modes=[('relative',False,False),('numerator_only',True,False),('denominator_only',False,True),('full_dirichlet',True,True)]
    for fold,(tr,va) in enumerate(splitter.split(texts,labels),1):
        vectorizer=CountVectorizer(ngram_range=(1,2),max_features=12000,dtype=np.float64)
        a=vectorizer.fit_transform(texts[tr]).toarray();b=vectorizer.transform(texts[va]).toarray()
        bg=a.sum(axis=0)/a.sum()
        for mu in mus:
            if mu<=0:raise ValueError('Use positive mu for the planned factorial contrast.')
            for mode,numerator,denominator in modes:
                model=RandomForestClassifier(n_estimators=100,max_depth=None,min_samples_split=2,min_samples_leaf=1,max_features='sqrt',class_weight='balanced',random_state=seed,n_jobs=1)
                model.fit(factorial_features(a,bg,mu,numerator,denominator),labels[tr])
                prediction=model.predict(factorial_features(b,bg,mu,numerator,denominator))
                rows.append(dict(fold=fold,mu=mu,mode=mode,macro_f1=f1_score(labels[va],prediction,average='macro',zero_division=0),zero_vocabulary_validation_rows=int((b.sum(axis=1)==0).sum())))
    result=pd.DataFrame(rows)
    target=Path('future_training_screening');target.mkdir(exist_ok=True)
    result.to_csv(target/'exploratory_cv_only.csv',index=False)
    print('Training-only screening completed. These scores are not independent test results.')
    return result

# Deliberately disabled. Enable only to perform additional exploratory train CV.
RUN_FUTURE_SCREENING=False
if RUN_FUTURE_SCREENING:
    display(run_training_screening())
else:
    print('Future factorial screening not run; archived results remain unchanged.')


In [ ]:
# Export computed evidence; Colab downloads the zip when available.
import shutil,json,hashlib
from pathlib import Path
out=Path('nlp_results')
required=['manifest.json','source_corpus.csv','clean_corpus.csv','corpus_split.csv','test_predictions.csv','classification_metrics.csv','per_class_metrics.csv','cv_candidates.csv','paired_comparison.csv','training_pearson_correlations.csv']
assert all((out/f).is_file() for f in required)
checksums={str(f.relative_to(out)):hashlib.sha256(f.read_bytes()).hexdigest() for f in out.rglob('*') if f.is_file() and f.name!='checksums.json'}
(out/'checksums.json').write_text(json.dumps(checksums,indent=2))
archive=shutil.make_archive('JIKI_RF_Smoothing_Evidence','zip',out)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Outside Colab: use the archive path above.')


## Writing assessment status

A Pangram analysis of the revised English manuscript prose was requested on November 17, 2026. The service returned an insufficient API-credit error and supplied no detection result. No detection score or claim of human authorship follows from this attempt. This status does not change the reported experiment results or replace editorial assessment.


## Submission versions

The identified manuscript retains the requested four-author order and proposed contribution statement for author review. The separate anonymous Word manuscript omits the author line, affiliations, contact details, and contribution statement, and clears author-related document properties for JIKI initial double-blind submission. The scientific content, references, equations, and reported measurements remain the same. The identified Markdown version retains authorship information.
